In [0]:
# ================================================================
# NOTEBOOK: nb_silver_orderitems_initial
# PURPOSE:  One-time full load Bronze → Silver for OrderItems
# RUN:      ONCE only — never run again after first execution
# SOURCE:   bronze/orderitems/  (parquet from ADF)
# TARGET:   silver/orderitems/  (Delta format)
# ================================================================



from pyspark.sql import functions as F
from pyspark.sql.functions import col, when,to_timestamp,round
from pyspark.sql.window import Window

BRONZE_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/bronze/orderitems/"
SILVER_PATH = "abfss://source@stshopsensedevhj.dfs.core.windows.net/silver/orderitems/"

# ── READ Bronze ───────────────────────────────────────────────
bronze_df = spark.read.parquet(BRONZE_PATH)
print(f"[BRONZE] Rows read:{bronze_df.count()} rows")
bronze_df.printSchema()

# ── STEP 1: Deduplication on OrderItemID ─────────────────────
# Keep latest record per OrderItemID
dedup_window = Window.partitionBy("OrderItemID").orderBy(F.desc("LastModifiedDate"))
bronze_df = (
    bronze_df
    .withColumn("_rn",F.row_number().over(dedup_window))
    .filter(col("_rn") ==1)
    .drop("_rn")
)

# ── STEP 2: Remove nulls on key columns ──────────────────
silver_df = (
    bronze_df
    .filter(col("OrderItemID").isNotNull())
    .filter(col("OrderID").isNotNull())
    .filter(col("ProductID").isNotNull())
    .filter(col("UnitPrice").isNotNull())
)
print(f"[CLEAN] After null removal:{silver_df.count()}")



# ── STEP 3: Type casting ──────────────────────────────────────

silver_df = (
    bronze_df
    .withColumn("LastModifiedDate",     to_timestamp("LastModifiedDate"))
    .withColumn("UnitPrice",            col("UnitPrice").cast("decimal(10,2)"))
    .withColumn("DiscountAmount",       col("DiscountAmount").cast("decimal(10,2)"))
    .withColumn("TotalPrice",       col("TotalPrice").cast("decimal(10,2)"))
    .withColumn("Quantity",       col("Quantity").cast("integer"))
    .withColumn("IsGift",           col("IsGift") == "True") 
)
# ── STEP 4: Derived / business columns ───────────────────────
silver_df = (
    silver_df
    # Effective price after discount
    .withColumn("EffectiveUnitPrice", round(col("UnitPrice") - (col("DiscountAmount") / col("Quantity")),2))

# Discount percentage (how much % was discounted)
    .withColumn("DiscountPct",
        when(col("UnitPrice") > 0,
        round(col("DiscountAmount") /
        (col("UnitPrice") * col("Quantity")) * 100,2))
        .otherwise(0.0))
    
# Net price after discount
    .withColumn("NetPrice",
                round(col("UnitPrice") * col("Quantity") - col("DiscountAmount"),2))
    
    
# Is this a discounted item?
    .withColumn("IsDiscounted", col("DiscountAmount") > 0)
    
# Gift item flag (already converted to boolean above)
# Category standardized

    .withColumn("Category", F.upper(F.trim(col("Category"))))

# Metadata

.withColumn("_silver_load_ts",  F.current_timestamp())
.withColumn("source",      F.lit("initial_full_load"))
.withColumn("_is_deleted",      F.lit(False))
)

# ── STEP 5: Write Silver as Delta ─────────────────────────────

(
silver_df.write
.format("delta")
.mode("overwrite")
.option("overwriteSchema", "true")
.save(SILVER_PATH)
)

# ── Verify ────────────────────────────────────────────────────
total    = silver_df.count()
gifted   = silver_df.filter(col("IsGift")      == True).count()
discount = silver_df.filter(col("IsDiscounted") == True).count()

print(f"[DONE] silver/orderitems/ written: {total} rows")
print(f"       Gift items:     {gifted}")
print(f"       Discounted:     {discount}")

# Step 1: Calculate the average as a small DataFrame
avg_result_df = silver_df.agg(F.avg('DiscountPct'))

# Step 2: Pull the value out of Spark into Python
avg_result_list = avg_result_df.collect()

# Step 3: Grab the actual number
avg_discount_value = avg_result_list[0][0]

# Step 4: Print with formatting
print(f"       Avg discount %: {avg_discount_value:.2f}%")

display(silver_df.limit(10))


[BRONZE] Rows read:5937 rows
root
 |-- OrderItemID: string (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- SellerID: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- DiscountAmount: decimal(10,2) (nullable = true)
 |-- TotalPrice: decimal(10,2) (nullable = true)
 |-- Category: string (nullable = true)
 |-- IsGift: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

[CLEAN] After null removal:5937
[DONE] silver/orderitems/ written: 5937 rows
       Gift items:     0
       Discounted:     3412
       Avg discount %: 3.75%


OrderItemID,OrderID,ProductID,SellerID,Quantity,UnitPrice,DiscountAmount,TotalPrice,Category,IsGift,LastModifiedDate,EffectiveUnitPrice,DiscountPct,NetPrice,IsDiscounted,_silver_load_ts,source,_is_deleted
ITEM00000001,ORD0000001,PROD0196,SELL015,4,14993.97,2249.10,57726.78,CLOTHING,false,2026-07-03T19:46:00.83Z,14431.70,3.75,57726.78,true,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000002,ORD0000002,PROD0247,SELL047,2,5880.31,0.00,11760.62,BEAUTY,false,2026-07-03T19:46:00.83Z,5880.31,0.0,11760.62,false,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000003,ORD0000003,PROD0009,SELL037,4,14504.86,725.24,55118.48,HOMEKITCHEN,false,2024-01-30T13:39:36Z,14323.55,1.25,57294.20,true,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000004,ORD0000003,PROD0241,SELL037,3,316.51,0.00,949.53,BOOKS,false,2026-07-03T19:46:00.83Z,316.51,0.0,949.53,false,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000005,ORD0000003,PROD0261,SELL037,4,10652.39,0.00,42609.56,BOOKS,false,2026-07-03T19:46:00.83Z,10652.39,0.0,42609.56,false,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000006,ORD0000004,PROD0195,SELL001,2,9454.95,1890.99,15127.92,HOMEKITCHEN,false,2024-03-02T10:25:54Z,8509.46,10.0,17018.91,true,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000007,ORD0000004,PROD0216,SELL001,1,8120.34,0.00,8120.34,HOMEKITCHEN,false,2024-03-02T10:25:54Z,8120.34,0.0,8120.34,false,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000008,ORD0000004,PROD0144,SELL001,3,2533.30,0.00,7599.90,BEAUTY,false,2024-03-02T10:25:54Z,2533.30,0.0,7599.90,false,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000009,ORD0000005,PROD0114,SELL009,1,13420.26,2684.05,10736.21,CLOTHING,false,2024-04-27T13:52:37Z,10736.21,20.0,10736.21,true,2026-07-09T19:41:24.371053Z,initial_full_load,false
ITEM00000010,ORD0000005,PROD0240,SELL009,3,13024.08,2604.82,31257.78,BOOKS,false,2024-04-27T13:52:37Z,12155.81,6.67,36467.42,true,2026-07-09T19:41:24.371053Z,initial_full_load,false
